# Policy gradients: teaching a lander to land

**The story of this notebook.** A lander has to learn to land, from rewards alone. We start with the simplest
policy-gradient method, REINFORCE, watch it fail, and ask *why*. Each answer is one idea from the lecture, and each
idea becomes the next method: a baseline, then a critic that judges every step, then a dial between the two (GAE),
then a way to use each batch of experience more than once (PPO). Every method is trained on the same task, with the
same seeds and the same budget, so each can be seen to beat the one before it on the training curves. Along the way, most parts run an
**experiment**: change one thing, keep the rest, and see what it does to learning.

**How to read it.** Cells marked `🎯 YOUR TURN` leave a few lines as `...`; the cell after each one checks your
answer. Cells marked `🔬 EXPERIMENT` run a controlled comparison and print what it found. Cells marked `⭐ BONUS` are
optional, for when you are ahead: nothing later depends on them, so skip them freely.
Each bonus has its own cell: set `RUN_BONUS = True` at its top, fill the lines marked 🎯 (if any), and run it.

In [ ]:
# Everything this notebook needs. If pip upgrades a package that was already imported, restart the kernel.
# Box2D, the physics of LunarLander, is built from source: it needs swig (installed first) and a C++ compiler.
%pip install -q swig
%pip install -q "gymnasium[box2d]>=1.1" "torch>=2.2" "numpy>=1.26" "matplotlib>=3.8" \
    "imageio>=2.31" "imageio-ffmpeg>=0.4.9" "pillow>=10.1" cloudpickle ipykernel

In [ ]:
import math
import os
import sys
import time
from pathlib import Path
from typing import Callable, Sequence
import warnings

warnings.filterwarnings("ignore", message="pkg_resources is deprecated")   # an old pygame warns at import...
os.environ["PYTHONWARNINGS"] = "ignore:pkg_resources is deprecated"       # ...here and in every worker process

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical, Distribution, Independent, Normal
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import display
if not Path("util.py").exists():   # on Colab: fetch the helper module from the course site
    import urllib.request
    urllib.request.urlretrieve("https://eth-cscs-hs26.github.io/CSCS-PAGE/exercises/part2/util.py", "util.py")
import util
from util import GRAY, PURPLE, PURPLE2, TEAL, GREEN, RED, AMBER, must_fill

ENV_ID = "LunarLander-v3"

def make_env(**kwargs) -> gym.Env:
    "A fresh copy of the environment (training in parallel needs many copies)."
    return gym.make(ENV_ID, **kwargs)

ACTION_SPACE = make_env().action_space
ACTION_SPACE.seed(0)            # so everyone's random actions match, as the environments' starts do
OBS_DIM = make_env().observation_space.shape[0]
TIME_LIMIT = make_env().spec.max_episode_steps
SOLVED = 200                    # a return of 200 counts as solving LunarLander
util.install_house_style()
torch.set_num_threads(1)        # tiny networks: one thread per process is fastest
TIMINGS: dict[str, float] = {}  # wall-clock seconds per part, printed at the end
RESULTS: dict[str, float] = {}  # final score per method, printed at the end
PART_T0 = time.perf_counter()
util.print_versions({"python": sys, "numpy": np, "torch": torch, "gymnasium": gym})

# Part 1 — The problem

A lander falls towards the Moon. It has a main engine underneath and two side engines, and it must come to rest on
the landing pad between the two flags, gently and upright. At every step we choose one of 4 actions.

- **State** $s_t$: 8 numbers — position ($x$, $y$), velocity, angle and angular velocity, and whether each leg
  touches the ground.
- **Action** $a_t$: do nothing, fire the left engine, fire the main engine, or fire the right engine.
- **Reward** $r_t$: points for moving towards the pad and slowing down, +10 for each leg that touches the ground,
  a small cost for every frame an engine fires, and at the end +100 for coming to rest or −100 for a crash.

The **return** of an episode, the sum of its rewards, is about −200 for a lander that crashes at random, and
**200 or more counts as solved**. An episode ends with a landing or a crash, or after 1,000 steps.

**The start and the goal.** On the left, random actions: the lander tumbles and crashes. On the right, the
controller that Gymnasium ships, written by hand from the physics (it steers towards a target angle and height),
which lands. In this notebook the computer has to find a policy like the one on the right **from the rewards
alone**, without knowing any physics.

In [ ]:
from gymnasium.envs.box2d.lunar_lander import heuristic

def random_act(obs):
    return ACTION_SPACE.sample()

env = make_env(render_mode="rgb_array")

def hand_act(obs):
    return heuristic(env.unwrapped, obs)           # Gymnasium's hand-written controller

clips, labels = [], []
for name, act in [("random actions", random_act), ("hand-written controller", hand_act)]:
    frames, ret = util.record_episode(env, act, seed=0, frame_every=3)
    clips.append(frames)
    labels.append(f"{name}: return {ret:.0f}")
display(util.show_video(util.tile(clips, labels), "start_and_goal.mp4", fps=17))

## 1.1 The agent–environment loop

Every interaction with the environment is the same loop: `reset()` starts an episode and returns the first state;
`step(action)` returns the next state, the reward, and two flags — `terminated` (the lander came to rest or
crashed) and `truncated` (the 1,000-step limit). Here is the start of one episode with random actions.

In [ ]:
env = make_env()
obs, info = env.reset(seed=0)
for t in range(1, 4):
    action = random_act(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"t={t}  action={action}  reward={reward:6.2f}  state={np.round(obs, 2)}  terminated={terminated}")
env.close()

**The return and the objective.** We weigh rewards that come later a little less, with a discount $\gamma = 0.99$.
The return from step $t$ is

$$G_t = r_t + \gamma\, r_{t+1} + \gamma^2 r_{t+2} + \dots + \gamma^{T-t} r_T .$$

The **policy** $\pi_w(a \mid s)$ is a neural network with weights $w$ that gives the probability of each action.
The goal is to choose $w$ so that the expected return $J(w) = \mathbb{E}[G_1]$ is as large as possible. We cannot
compute $J$ exactly, but the average return $\bar G$ of a batch of episodes estimates it.

(The lecture uses $\gamma = 1$; with $\gamma < 1$ the update below is the one used in practice and in the papers.
Code counts steps from 0, the formulas from 1.)

**Measuring the score.** Before training anything, we need a yardstick. The number we will watch in every part is
the **score** of a policy: the average, over several episodes, of the plain sum of an episode's rewards,
$\bar G = \frac1n \sum_i \sum_t r^{(i)}_t$. That sum has **no discount**, so it is the return with
$\gamma = 1$, and it is the number Gymnasium reports and that "200 counts as solved" refers to. The discount
$\gamma = 0.99$ appears only inside the *training signal* $G_t$ (Part 2); scores are always measured without it.

The cell below plays 20 episodes (seeds 0 to 19) with each of the two controllers from the videos above and prints
their scores. The random controller sets the floor and the hand-written one the ceiling that learning has to reach.

In [ ]:
env_plain = make_env()
for name, act in [("random actions", random_act), ("hand-written controller", lambda o: heuristic(env_plain.unwrapped, o))]:
    returns = []
    for ep in range(20):
        obs, _ = env_plain.reset(seed=ep)
        done, R = False, 0.0
        while not done:
            obs, r, term, trunc, _ = env_plain.step(act(obs))
            R += r
            done = term or trunc
        returns.append(R)
    print(f"{name:24s} G-bar over 20 episodes = {np.mean(returns):6.1f}   (solved: {SOLVED})")
TIMINGS["1 The problem"] = time.perf_counter() - PART_T0

# Part 2 — REINFORCE (Algorithm 1)

**The idea.** Play some episodes. Then make every action taken in an episode more likely, in proportion to how
well the episode went from that action on. The number that says "how well" is called the **multiplier** $\Psi_t$.
In REINFORCE it is the return that followed the action: $\Psi_t = G_t$.

**The key equation.** After a batch of $n$ episodes, the loop takes a gradient step on

$$\tilde J(w) = \frac1n \sum_{i=1}^{n} \sum_{t=1}^{T_i} \Psi^{(i)}_t \,\log \pi_w\!\left(a^{(i)}_t \mid s^{(i)}_t\right),$$

with the multipliers held fixed. $\tilde J$ is **not** the objective $J$ and not a score: it is a *surrogate*, a number
built from one batch whose **gradient** is what we want. The lecture's "log trick" turns $\nabla_w J$ into an average
of multiplier times **score** $\nabla_w \log\pi_w(a_t \mid s_t)$; $\tilde J$ is that average written so that
autograd can differentiate it. Its gradient is therefore an estimate of $\nabla_w J$: noisy for one batch, right on
average (the lecture's "What goes into the gradient box" frame shows both on Nim).

In [ ]:
PART_T0 = time.perf_counter()
SEEDS = [1, 2, 3, 4, 5]                           # fixed seeds: every training and experiment below uses these
# SEEDS = util.random_seeds(5)                   # or fresh random seeds, drawn once here
N_STEPS = 400_000                             # environment steps per seed, the same budget for every method
EXP_STEPS = 300_000                           # the budget of each experiment run (several settings per experiment)
EXP_STEPS_SHORT = 200_000                     # for Part 2's experiments: REINFORCE's settings separate early
EVAL_EVERY = 25_000                           # evaluate the policy every 25,000 environment steps
GAMMA = 0.99
print("seeds of this run:", SEEDS, "(to repeat a run exactly, set SEEDS to this list by hand)")

## 2.1 The policy network

$\pi_w$ is a small network (`util.mlp`: two hidden layers of 64 ReLU units). It maps the 8 numbers of the state to
4 scores, one per action, and `Categorical` turns them into probabilities with a softmax. Calling `policy(state)`
returns a **distribution** with the three methods we need:
- `dist.sample()` draws an action, for training;
- `dist.log_prob(action)` gives $\log\pi_w(a \mid s)$, differentiable in $w$;
- `dist.mode` is the most likely action (Bonus 1 (c) shows what it does).

The last layer starts near zero, so the untrained policy picks each action with probability about 0.25. (The same
class also handles continuous actions with a Gaussian; Bonus 1 uses that.)

In [ ]:
class Policy(nn.Module):
    "pi_w: states -> a distribution over actions (Categorical for discrete actions, Gaussian for continuous)."

    def __init__(self, obs_dim: int, action_space: gym.Space, hidden: int = 64):
        super().__init__()
        self.discrete = isinstance(action_space, gym.spaces.Discrete)
        n_out = action_space.n if self.discrete else action_space.shape[0]
        self.net = util.mlp(obs_dim, n_out, hidden, out_gain=0.01)
        if not self.discrete:
            self.log_std = nn.Parameter(torch.full((n_out,), -0.5))

    def forward(self, obs: torch.Tensor) -> Distribution:
        out = self.net(obs)
        if self.discrete:
            return Categorical(logits=out)
        return Independent(Normal(out, self.log_std.exp()), 1)


def to_env_action(action: torch.Tensor, space: gym.Space = ACTION_SPACE):
    "A torch action -> what env.step expects (an int, or a float array within the bounds)."
    if isinstance(space, gym.spaces.Discrete):
        return action.item() if action.ndim == 0 else action.numpy()
    return np.clip(action.numpy(), space.low, space.high)

In [ ]:
torch.manual_seed(SEEDS[0])
demo_policy = Policy(OBS_DIM, ACTION_SPACE)
s, _ = make_env().reset(seed=0)
dist = demo_policy(torch.as_tensor(s, dtype=torch.float32))
a = dist.sample()
print("probabilities of nothing, left, main, right:", dist.probs.detach().numpy().round(3))
print(f"sampled action {a.item()}, log π_w(a | s) = {dist.log_prob(a).item():.3f}  (log 0.25 = -1.386)")

## 2.2 The return of every step

$G_t$ needs every reward after step $t$, so we compute it **backwards**, starting from the last step:

$$G_T = r_T, \qquad G_t = r_t + \gamma\, G_{t+1}.$$

Example with rewards 1, 1, 1 and $\gamma = 0.5$: $G_3 = 1$, then $G_2 = 1 + 0.5 \cdot 1 = 1.5$, then
$G_1 = 1 + 0.5 \cdot 1.5 = 1.75$.

In [ ]:
# 🎯 YOUR TURN — Exercise 1: the return of every step.
#
# 🎯 Walk the episode backwards: G_T = r_T, then G_t = r_t + γ G_{t+1}. One line.

def discounted_returns(rewards: Sequence[float], gamma: float) -> np.ndarray:
    "G_t for every step of one episode, computed backwards from the last step."
    G, out = 0.0, np.zeros(len(rewards))
    for t in reversed(range(len(rewards))):
        G = ...
        out[t] = G
    return out

In [ ]:
try:
    _G = discounted_returns([1.0, 1.0, 1.0], gamma=0.5)
except TypeError:            # `...` is still in the loop
    _G = ...
must_fill(discounted_returns=_G)
assert np.allclose(_G, [1.75, 1.5, 1.0]), f"got {_G}, expected [1.75, 1.5, 1.0]"
assert np.allclose(discounted_returns([1.0] * 4, 1.0), [4, 3, 2, 1]), "with gamma = 1, G_t counts the steps left"
print("  the returns are correct ✓")

## 2.3 Playing one episode

`run_episode` plays the policy until the episode ends. It keeps the states, the rewards and the log-probabilities
of the actions taken. The log-probabilities keep their link to $w$ (`requires_grad=True`), so that after the episode
we can differentiate the loss with respect to $w$.

In [ ]:
def run_episode(env: gym.Env, policy: Policy, seed: int | None = None):
    "Play one episode with `policy`; return the states, the log-probabilities (with grad) and the rewards."
    obs, _ = env.reset(seed=seed)
    states, log_probs, rewards = [], [], []
    while True:
        s = torch.as_tensor(obs, dtype=torch.float32)
        dist = policy(s)
        action = dist.sample()
        states.append(s)
        log_probs.append(dist.log_prob(action))
        obs, r, terminated, truncated, _ = env.step(to_env_action(action))
        rewards.append(float(r))
        if terminated or truncated:
            return torch.stack(states), torch.stack(log_probs), rewards

In [ ]:
_states, _lp, _rew = run_episode(make_env(), demo_policy, seed=1)
_G = discounted_returns(_rew, GAMMA)
print(f"one episode of the untrained policy lasted {len(_rew)} steps")
print(f"  its first log-probabilities: {_lp[:4].detach().numpy().round(3)} (requires_grad={_lp.requires_grad})")
print(f"  its returns: G_1 = {_G[0]:.2f} ... G_T = {_G[-1]:.2f}  → the crash at the end weighs on every action before it")

## 2.4 The loss

The optimiser *minimises*, so we give it $-\tilde J$. For one episode the loss is

$$-\sum_{t=1}^{T} G_t \, \log\pi_w(a_t \mid s_t),$$

and the training loop averages it over the $n$ episodes of a batch. $G_t$ is a plain number here, the boxed $\Psi$ of
the lecture's formulas: it weighs each log-probability, and no gradient may flow through it (hence `G.detach()`).
If we differentiated through $G_t$ we would be differentiating something else, not an estimate of $\nabla_w J$.

Do not read the loss's *value* as progress: it is minus a surrogate that changes with every batch. What improves is
the **score** of Section 1, the average return. The loss matters only through its gradient.

In [ ]:
# 🎯 YOUR TURN — Exercise 2: the REINFORCE loss of one episode.
#
# 🎯 Return -Σ_t G_t log π_w(a_t | s_t). Detach G first: the multiplier is held fixed.

def reinforce_loss(log_probs: torch.Tensor, G: torch.Tensor) -> torch.Tensor:
    "Minus J-tilde for one episode: -sum_t G_t log pi_w(a_t | s_t)."
    return ...

In [ ]:
_lp = torch.tensor([-0.5, -1.0, -2.0], requires_grad=True)
_G = torch.tensor([3.0, 2.0, 1.0])
_loss = reinforce_loss(_lp, _G)
must_fill(reinforce_loss=_loss)
assert torch.isclose(_loss, torch.tensor(5.5)), f"got {_loss.item()}, expected -(3·-0.5 + 2·-1 + 1·-2) = 5.5"
_loss.backward()
assert torch.allclose(_lp.grad, -_G), "d loss / d log π_t should be -G_t"
print("  the REINFORCE loss is correct ✓")

## 2.5 Evaluation

The training returns are noisy measurements: every episode starts from a different position, and the policy
changes while a batch is collected. To see what a policy has learned, we also **evaluate** it now and then: freeze
it, play the same 10 test episodes (fixed starting positions), sampling its actions as training does, and average
their returns. That is an estimate of $J(w)$, the objective the methods maximise. `sampled` turns a policy into such
an action function, and `final_score` sums up a whole run by the mean of its last three evaluations.

A natural alternative is to take the most likely action every time, which is how a finished policy is often
deployed. Bonus 1 (c), at the end of this part, writes that action function and shows why it is a poor way to judge
a policy that is still undecided.

In [ ]:
def sampled(policy: Policy) -> Callable:
    "An action function that samples from the policy, as training does."
    def act(obs):
        with torch.no_grad():
            return to_env_action(policy(torch.as_tensor(obs, dtype=torch.float32)).sample())
    return act


def evaluate(act: Callable, seed: int, n: int = 10) -> float:
    "Mean return of n episodes played with the action function `act`, from the fixed starts seed, ..., seed + n - 1."
    env, returns = make_env(), []
    with torch.random.fork_rng(devices=[]):      # its own random numbers: training's are left exactly as they were
        torch.manual_seed(seed)
        for i in range(n):
            obs, _ = env.reset(seed=seed + i)
            done, R = False, 0.0
            while not done:
                obs, r, term, trunc, _ = env.step(act(obs))
                R += r
                done = term or trunc
            returns.append(R)
    return float(np.mean(returns))


def final_score(run: dict) -> float:
    "A run's final performance: the mean of its last three evaluations (one evaluation is noisy)."
    return float(np.mean([v for _, v in run["evals"][-3:]]))

## 2.6 The training loop

Algorithm 1 has four lines, and each is one line of `reinforce`:

| Algorithm 1 | Code |
|---|---|
| 1. Collect $n$ complete episodes with $\pi_w$. | `run_episode(...)`, `batch_size` times |
| 2. Compute every $G^{(i)}_t$. | `discounted_returns(...)` |
| 3. $w \leftarrow w + \alpha\,\nabla_w\tilde J$ | the average of `reinforce_loss(...)`, then `backward()` and `opt.step()` |
| 4. Discard the batch; repeat. | the next turn of the `while` loop |

We use batches of $n = 8$ episodes and the Adam optimiser with step size $\alpha = 10^{-3}$; 2.9 tests both
choices. Every method of this notebook gets the same budget, counted in **environment steps** (episodes have very
different lengths here), and is evaluated every 25,000 steps. The function returns the training return of every
episode with the step at which it ended, the evaluations, and the trained weights.

In [ ]:
def reinforce(seed: int, total_steps: int, batch_size: int = 8, lr: float = 1e-3, gamma: float = 0.99,
              eval_every: int = 25_000) -> dict:
    "Algorithm 1: train a fresh policy with REINFORCE; return its training returns, evaluations and weights."
    torch.manual_seed(seed)
    env = make_env()
    env.reset(seed=seed)          # seed the environment once; every later reset continues its random stream
    policy = Policy(env.observation_space.shape[0], env.action_space)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    returns, steps_at, evals, steps, next_eval = [], [], [], 0, eval_every
    while steps < total_steps:
        loss = 0.0
        for _ in range(batch_size):                                        # 1. collect n episodes
            _, log_probs, rewards = run_episode(env, policy)
            G = torch.as_tensor(discounted_returns(rewards, gamma), dtype=torch.float32)   # 2. returns
            loss = loss + reinforce_loss(log_probs, G) / batch_size        # 3. minus J-tilde ...
            steps += len(rewards)
            returns.append(sum(rewards))
            steps_at.append(steps)
        opt.zero_grad()
        loss.backward()
        opt.step()                                                         #    ... one gradient step
        while steps >= next_eval:                                          # 4. discard, repeat
            evals.append((next_eval, evaluate(sampled(policy), seed=10**6 + next_eval)))
            next_eval += eval_every
    return {"seed": seed, "returns": returns, "steps_at": steps_at, "evals": evals, "weights": policy.state_dict()}

## 2.7 How noisy is the estimate?

Before training, look at the gradient estimate itself. Freeze the untrained policy. As our best guess of the true
gradient, average a few estimates that each use a large batch (48 episodes). Then compute many estimates from
**1**, **8** and **32** episodes, and compare each with that guess by their **cosine**: 1 if the two gradients point
the same way, 0 if they are unrelated, −1 if they point in opposite directions. In the plot, each dot is one estimate,
and a **red** dot is an estimate that would push the policy the wrong way. The cell plays about a thousand episodes,
so it takes a minute or two.

In [ ]:
def gradient_estimates(policy: Policy, n_estimates: int, batch_size: int) -> torch.Tensor:
    "n_estimates REINFORCE gradients, each averaged over batch_size episodes; shape (n_estimates, n_weights)."
    env, grads = make_env(), []
    env.reset(seed=0)
    for _ in range(n_estimates):
        policy.zero_grad()
        for _ in range(batch_size):
            _, log_probs, rewards = run_episode(env, policy)
            G = torch.as_tensor(discounted_returns(rewards, GAMMA), dtype=torch.float32)
            (reinforce_loss(log_probs, G) / batch_size).backward()
        grads.append(torch.cat([p.grad.flatten() for p in policy.parameters()]))
    return torch.stack(grads)

torch.manual_seed(SEEDS[0])
probe = Policy(OBS_DIM, ACTION_SPACE)
reference = gradient_estimates(probe, 6, 48).mean(0, keepdim=True)   # our best guess of the true gradient
batch_sizes = [1, 8, 32]
jitter = np.random.default_rng(0)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.axhspan(-1, 0, color=RED, alpha=0.08)                              # below zero: a step in the wrong direction
ax.axhline(0, color="black", lw=0.8)
for k, n in enumerate(batch_sizes):
    cos = nn.functional.cosine_similarity(gradient_estimates(probe, 20, n), reference).numpy()
    ax.scatter(k + jitter.uniform(-0.25, 0.25, len(cos)), cos, s=22, alpha=0.85, zorder=3,
               c=[RED if v < 0 else GREEN for v in cos])
    ax.hlines(np.median(cos), k - 0.32, k + 0.32, color="black", lw=2, zorder=4)
    ax.text(k, 1.06, f"{(cos < 0).mean():.0%} wrong way", ha="center", fontsize=9)
ax.set_xticks(range(len(batch_sizes)), [f"{n} episode{'s' if n > 1 else ''}" for n in batch_sizes])
ax.set_xlim(-0.5, len(batch_sizes) - 0.5)
ax.set_ylim(-1, 1.18)
ax.set_xlabel("episodes averaged into one gradient estimate")
ax.set_ylabel("cosine with the reference gradient")
ax.set_title("One dot per gradient estimate (black bar: median)")
plt.show()

**The variance problem.** A gradient computed from one episode often points the wrong way. A batch of 8 helps, and a
batch of 32 helps much more, but a larger batch is costly: each step then needs many more episodes (2.9 tests the
batch size). On average the estimate is right; any single one is noisy, so learning is
slow and differs a lot from one seed to the next. This is the problem the rest of the notebook works on.

## 2.8 Training and results

We train one policy per seed. The seeds are independent, so they run **in parallel**, one process per CPU core:
`util.parallel_map` starts fresh Python processes and sends each one the function to run (here, `reinforce` with
its seed) — the same pattern as a parameter sweep on a cluster.

In [ ]:
t0 = time.perf_counter()
reinforce_runs = util.parallel_map(lambda seed: reinforce(seed, N_STEPS, gamma=GAMMA, eval_every=EVAL_EVERY), SEEDS)
print(f"{len(SEEDS)} seeds × {N_STEPS:,} steps in {time.perf_counter() - t0:.0f} s")
for run in reinforce_runs:
    print(f"  seed {run['seed']:>5}: final score {final_score(run):7.0f}, "
          f"G-bar of the last 50 training episodes {np.mean(run['returns'][-50:]):7.0f}")

Left, the training return of every episode (a 20-episode average); right, the evaluations. The line is the mean
over the seeds and the band runs from the worst seed to the best. The x axis counts environment steps. Then a video
of each seed's final policy, sampling its actions as it trained, as every video of the notebook does (Bonus 1 (c)
shows what happens when the same policy always takes its most likely action instead).

**What to look for.** REINFORCE alone usually does not land in this budget: both curves rise slowly and stay far
below 200, and the seeds differ a lot. The evaluations replay the same test episodes, so they are steadier than the
training returns.

In [ ]:
def plot_methods(methods: dict, title: str) -> None:
    "Training and evaluation returns of several methods, each as a mean line and a best-to-worst-seed band."
    fig, (ax_t, ax_e) = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
    for name, (runs, color) in methods.items():
        util.plot_runs(ax_t, ax_e, runs, color, name, x_key="steps")
    util.finish_return_axes(ax_t, ax_e, "environment steps", SOLVED)
    fig.suptitle(title)
    plt.show()


def show_policies(runs: list, path: str, act_maker: Callable = sampled) -> None:
    "One episode of each seed's final policy, side by side, its actions sampled (or `act_maker` another rule)."
    env, clips, labels = make_env(render_mode="rgb_array"), [], []
    for run in runs:
        policy = Policy(OBS_DIM, ACTION_SPACE)
        policy.load_state_dict(run["weights"])
        frames, ret = util.record_episode(env, act_maker(policy), seed=123, frame_every=3)
        clips.append(frames)
        labels.append(f"seed {run['seed']}: return {ret:.0f}")
    display(util.show_video(util.tile(clips, labels, cols=min(3, len(clips)), gap=6), path, fps=17, width=960))


plot_methods({"REINFORCE": (reinforce_runs, AMBER)}, "REINFORCE on LunarLander")
show_policies(reinforce_runs, "reinforce.mp4")
RESULTS["REINFORCE"] = float(np.mean([final_score(r) for r in reinforce_runs]))

## 2.9 🔬 Experiment: batch size and step size

Two knobs that every policy-gradient method has, and that the lecture's Algorithm 1 leaves to us: the number of
episodes $n$ averaged into one gradient, and the step size $\alpha$. Section 2.7 suggests that a larger $n$ gives a
better gradient; but with a fixed budget of environment steps, a larger $n$ also means *fewer* gradient steps.

`experiment` is the tool for every experiment of this notebook: it trains every setting on the same seeds, in
parallel, with a smaller budget than the main runs (`EXP_STEPS`), plots the evaluations with their seed bands, and
prints each setting's final score. Change **one** thing at a time, and read the band, not only the line: with three
seeds, two settings whose bands overlap are not distinguishable.

**Questions to answer from the plot.** (1) Does $n = 1$ learn at all? (2) The largest $n$ gets the best gradient
per update but the fewest updates in the budget: which wins? (3) What does a step size ten times larger, or ten
times smaller, do? With a handful of seeds and a policy this noisy, read the
bands as well as the lines.

In [ ]:
def experiment(train: Callable, settings: dict, title: str, steps: int | None = None, seeds: list | None = None,
               reference: dict | None = None) -> dict:
    '''Train every setting of `settings` ({label: kwargs}) on every seed, in parallel; plot and print the results.

    `train(seed, total_steps, **kwargs)` must return a run dict with "evals". Returns {label: [runs]}.
    `reference`, if given, is {label: [runs]} plotted beside the new ones (for a run already done).
    '''
    steps, seeds = steps or EXP_STEPS, seeds or SEEDS
    jobs = [(label, seed) for label in settings for seed in seeds]
    t0 = time.perf_counter()
    results = util.parallel_map(lambda job: train(job[1], steps, eval_every=EVAL_EVERY, **settings[job[0]]), jobs)
    runs = {label: [r for (l, _), r in zip(jobs, results) if l == label] for label in settings}
    print(f"{len(jobs)} runs of {steps:,} steps in {time.perf_counter() - t0:.0f} s")
    fig, (ax_t, ax_e) = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
    palette = [PURPLE2, AMBER, GREEN, TEAL, GRAY, RED]
    for (label, rs), color in zip({**(reference or {}), **runs}.items(), palette):
        util.plot_runs(ax_t, ax_e, rs, color, label, x_key="steps")
    util.finish_return_axes(ax_t, ax_e, "environment steps", SOLVED)
    fig.suptitle(title)
    plt.show()
    for label, rs in runs.items():
        scores = [final_score(r) for r in rs]
        print(f"  {label:36s} final score {np.mean(scores):7.0f}   per seed {[round(s) for s in scores]}")
    return runs


batch_runs = experiment(reinforce, {"n = 1 episode per update": dict(batch_size=1),
                                    "n = 4": dict(batch_size=4),
                                    "n = 8 (ours)": dict(batch_size=8),
                                    "n = 32": dict(batch_size=32)},
                        "REINFORCE: episodes per gradient step, same budget of environment steps", steps=EXP_STEPS_SHORT)

In [ ]:
lr_runs = experiment(reinforce, {"α = 1e-4": dict(lr=1e-4), "α = 1e-3 (ours)": dict(lr=1e-3), "α = 1e-2": dict(lr=1e-2)},
                     "REINFORCE: step size, same budget of environment steps", steps=EXP_STEPS_SHORT)

**Reading the experiment.** The batch size trades the quality of each gradient against the number of gradients
the budget buys. At this budget the many noisy steps of small batches usually win: one episode per update tends to
learn fastest, and 32 episodes per update, which take the fewest steps, slowest. The price of small batches shows in
the bands, which are widest for one episode. The step size trades speed against stability in the same way. Whatever the setting, the training curves stay far
from 200: neither knob removes the noise itself — the gradient of one batch still points the wrong way often — and
that is what Part 3 is about.

## ⭐ Bonus 1 — Reward-to-go, the continuous lander, and the most likely action

*Nothing later depends on this cell.*

**(a) Reward-to-go.** The lecture's $\tilde J$ multiplies each action by the return *from that step on*, $G_t$.
A simpler estimator multiplies every action of an episode by the *whole* episode's return, $G_1$. Both are
unbiased. Which is noisier, and why? The first cell below tests it: `reinforce_loss_whole` uses `G[0]`
for every step and is swapped into `reinforce`, then both are compared on one seed, with the main run's budget (about 5 minutes).

**(b) The continuous lander.** `gym.make("LunarLander-v3", continuous=True)` replaces the four buttons by two
throttles in $[-1, 1]$. `Policy` already handles it: the network outputs the mean of a Gaussian per throttle, with a
learned spread. The second cell below runs `reinforce` on it (same switch and budget, about 5 minutes). Does the discrete or the continuous version learn
faster for the same budget, and what does the learned spread (`policy.log_std`) do over training?

In [ ]:
# ⭐ Bonus 1 (a) — reward-to-go against the whole-episode return (about 5 minutes; same budget as the main run).
RUN_BONUS = False
if RUN_BONUS:
    def reinforce_loss_whole(log_probs, G):
        return -(G[0].detach() * log_probs).sum()           # every action weighed by the whole episode's return

    def reinforce_whole(seed, total_steps, **kw):
        global reinforce_loss
        keep, reinforce_loss = reinforce_loss, reinforce_loss_whole
        try:
            return reinforce(seed, total_steps, **kw)
        finally:
            reinforce_loss = keep

    experiment(reinforce_whole, {"whole-episode return G_1": {}}, "REINFORCE with G_1 instead of G_t",
               steps=N_STEPS, seeds=SEEDS[:1], reference={"reward-to-go G_t (ours)": reinforce_runs[:1]})
TIMINGS["2 REINFORCE"] = time.perf_counter() - PART_T0

In [ ]:
# ⭐ Bonus 1 (b) — a starting point: REINFORCE on the continuous lander (same RUN_BONUS switch as above; about 5 minutes).
if RUN_BONUS:
    def make_env_continuous(**kwargs):
        return gym.make(ENV_ID, continuous=True, **kwargs)        # two throttles in [-1, 1] instead of four buttons

    def reinforce_continuous(seed, total_steps, **kw):
        global make_env
        keep, make_env = make_env, make_env_continuous             # Policy, run_episode and evaluate all use make_env
        try:
            return reinforce(seed, total_steps, **kw)
        finally:
            make_env = keep

    continuous_runs = experiment(reinforce_continuous, {"continuous (Gaussian policy)": {}},
                                 "REINFORCE on the continuous lander", seeds=SEEDS[:1],
                                 steps=N_STEPS, reference={"discrete (four buttons)": reinforce_runs[:1]})
    spread = continuous_runs["continuous (Gaussian policy)"][0]["weights"]["log_std"].exp()
    print("learned spread (standard deviation) of the main and the side throttle:", spread.numpy().round(2),
          "  (it starts at", round(float(np.exp(-0.5)), 2), ")")

**Takeaways of Bonus 1.**
- **(a)** Reward-to-go should be the better estimator. $G_1$ also credits each action with the rewards that came
  *before* it, which the action could not have influenced: those terms add noise to the gradient and no signal, while
  both estimators stay unbiased. With a single seed this shows, if at all, on the curves. Repeat with more seeds before
  concluding.
- **(b)** The algorithm does not change: the loss is still $-\sum_t G_t \log\pi_w(a_t \mid s_t)$. Only the
  distribution does, from a softmax over four buttons to a Gaussian over two throttles, whose log-probability is
  differentiable in its mean and its spread. Compare the printed spread with its starting value: expect it to move
  little at this budget, because the policy's randomness changes much more slowly than its mean. Which version
  learns faster is for you to read off the plot.

### ⭐ Bonus 1 (c) — The most likely action, and why the notebook does not evaluate with it

*Nothing later depends on these cells.*

A finished policy is often deployed by taking its most likely action at every step (`greedy`). For REINFORCE at this
budget, that gives a very different, and much worse, answer than sampling. That is not a bug in `greedy`, which
takes exactly the most likely action. It comes from what REINFORCE trains.

1. REINFORCE improves the policy that **samples** its actions: $J(w)$ is the expected return of *that* policy. At
   this budget its probabilities are still flat. On a fresh episode the best action is usually the main engine, but
   often by a small margin over the others.
2. **Greedy turns a small edge into a certainty.** Where the sampled lander fires the main engine on a share of the
   steps, the greedy one fires it on every step. The sampled policy produces a gentle average thrust and settles; the
   greedy policy produces a constant, larger one, and the lander rises.
3. **Then it is somewhere the policy has never played.** Far above anything seen in training, the probabilities mean
   nothing, and greedy goes on firing the same way until the time limit, paying for fuel and drifting away.

First write that action function, in the first cell below; the second checks it. Then the third cell checks the story on the final policy of every seed. The table compares the two ways of acting; the left
plot adds two baselines, always firing the main engine and uniformly random actions; the right plot follows the
height of the lander. The greedy version does worse than random.

In [ ]:
# ⭐ Bonus 1 (c) — the greedy action: the policy's most likely action, not a sample from it. Fill the one 🎯 line
#    (use `dist.mode`).

def greedy(policy: Policy) -> Callable:
    "An action function that always takes the policy's most likely action."
    def act(obs):
        with torch.no_grad():
            dist = policy(torch.as_tensor(obs, dtype=torch.float32))
            action = ...                       # 🎯 the most likely action
            return to_env_action(action)
    return act

In [ ]:
# ⭐ Bonus 1 (c) — check your greedy action.
RUN_BONUS = False
if RUN_BONUS:
    try:
        _act = greedy(lambda obs: Categorical(logits=torch.tensor([0.1, 2.0, 0.3, 0.5])))   # action 1 is the most likely
        _a = [_act(np.zeros(8)) for _ in range(200)]
    except AttributeError:            # `...` is still in the function
        _a = ...
    must_fill(_where="the cell above", greedy=_a)
    assert all(a == 1 for a in _a), f"greedy took actions {sorted(set(_a))}, but action 1 is always the most likely"
    print("  the greedy action is correct ✓")

In [ ]:
# ⭐ Bonus 1 (c) — sampled against greedy, on the final policy of every seed (about a minute).
RUN_BONUS = False
if RUN_BONUS:
    def height_trace(act: Callable, seed: int) -> list:
        "The lander's height (the second state number) at every step of one episode."
        env = make_env()
        obs, _ = env.reset(seed=seed)
        heights, done = [obs[1]], False
        while not done:
            obs, _, term, trunc, _ = env.step(act(obs))
            heights.append(obs[1])
            done = term or trunc
        return heights


    torch.manual_seed(0)                                         # the same sampled actions at every run
    print("mean return of the final policy of each seed, over 6 episodes")
    print(f"{'seed':>8}  {'sampled':>9}  {'greedy':>9}")
    for run in reinforce_runs:
        p = Policy(OBS_DIM, ACTION_SPACE)
        p.load_state_dict(run["weights"])
        print(f"{run['seed']:>8}  {evaluate(sampled(p), seed=900, n=6):9.0f}  {evaluate(greedy(p), seed=900, n=6):9.0f}")

    policy0 = Policy(OBS_DIM, ACTION_SPACE)                      # the first seed's policy, acted out four ways
    policy0.load_state_dict(reinforce_runs[0]["weights"])
    ways = {"sampled": (sampled(policy0), GREEN), "greedy": (greedy(policy0), RED),
            "always main engine": (lambda o: 2, GRAY), "random": (random_act, GRAY)}
    fig, (ax_bar, ax_h) = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1, 1.3]}, layout="constrained")
    means = [evaluate(act, seed=900, n=6) for act, _ in ways.values()]
    ax_bar.barh(list(ways), means, color=[c for _, c in ways.values()])
    ax_bar.axvline(0, color="black", lw=0.8)
    ax_bar.invert_yaxis()
    for y, m in enumerate(means):
        ax_bar.text(0, y, f"  {m:.0f}", va="center", ha="left", fontsize=10)            # the numbers sit right of zero
    ax_bar.set_xlim(min(means) * 1.05, -min(means) * 0.2)
    ax_bar.set_xlabel("mean return over 6 episodes")
    ax_bar.set_title("First seed's policy, acted out four ways")
    for k in range(3):
        ax_h.plot(height_trace(greedy(policy0), seed=900 + k), color=RED, alpha=0.8, label="greedy" if k == 0 else None)
        ax_h.plot(height_trace(sampled(policy0), seed=900 + k), color=GREEN, alpha=0.8, label="sampled" if k == 0 else None)
    ax_h.set_xlabel("step of the episode")
    ax_h.set_ylabel("height of the lander (it starts at 1.4)")
    ax_h.set_title("Where the lander goes")
    ax_h.legend(frameon=False)
    plt.show()

**The lesson.** A stochastic policy is not the same thing as its most likely action. Taking the most likely action is
the standard way of using a *converged* policy, whose probabilities are close to 0 or 1. It is a poor way of judging
a policy that is still undecided. That is why this notebook evaluates by sampling. From Part 4 on, the
policies get sharper and the two ways of acting agree more closely.

**Nicer here, not always the best choice.** For this lander the stochastic policy is the friendlier one. Sampling is
how it explores while it learns, and at the end the randomness is forgiving: a thrust that is a little off in one
step is corrected in the next, and the mixed firing averages into a gentle descent. Precise control is less
forgiving. A robot arm that must stop within a millimetre, a vehicle that must land the same way every time, or a
controller that has to be certified needs the *same* action in the *same* state, and dice rolled at every step add an
error that never averages out. There we want a policy that is (almost) deterministic once training is over: one
trained until its probabilities are sharp, so that its most likely action is overwhelmingly likely; or, for
continuous actions, the mean of its Gaussian instead of a sample from it; or a method that learns a deterministic
policy directly. Whether to sample or to take the most likely action is a design decision, not a detail of
evaluation.

# Part 3 — REINFORCE with a baseline (Algorithm 2)

**The problem, seen in Part 2.** $G_t$ mostly reflects *where* the lander was, not what it did: from high up and
far from the pad, the return is low whatever the action; from just above the pad, it is high. So the multiplier
pushes every action of a good episode up and every action of a bad one down, without asking: *was this action
better than usual, from here?* That is one reason the gradient of 2.7 points the wrong way so often.

**The idea.** Subtract what usually happens from this state. Call it $V(s_t)$: the return the policy gets on
average from $s_t$. Then

$$\Psi_t = G_t - V(s_t)$$

is positive when the episode went better than usual from $s_t$, and negative when it went worse. Good actions are
pushed up and bad ones down. This does not change the gradient on average (the subtracted part does not depend on
the action), but it removes much of the noise.

**Where $V$ comes from.** Nobody knows $V(s)$, so a second network, the **value network** $V_\phi(s)$ with its own
weights $\phi$, learns it: for every state of the batch, it is trained to predict the return $G_t$ that followed.
That is ordinary least-squares regression. The policy network is trained as before, but on the new multiplier.

In [ ]:
PART_T0 = time.perf_counter()

## 3.1 Two losses, one per network

For one episode:
- **policy loss:** $-\sum_t \bigl(G_t - V_\phi(s_t)\bigr)\log\pi_w(a_t \mid s_t)$. The multiplier is a fixed number, so $V$ is
  **detached** here: the policy loss must not change the value network.
- **value loss:** $\sum_t \bigl(G_t - V_\phi(s_t)\bigr)^2$, with $V$ **not** detached: this is the loss that trains the value network.

In [ ]:
# 🎯 YOUR TURN — Exercise 3: the losses of Algorithm 2 for one episode.
#
# 🎯 psi = G - V with V detached; policy loss = -Σ psi · log π; value loss = Σ (G - V)² with V NOT detached.

def baseline_losses(log_probs: torch.Tensor, G: torch.Tensor, V: torch.Tensor):
    "Return (policy loss, value loss) for one episode, with the multiplier Psi_t = G_t - V_phi(s_t)."
    psi = ...
    policy_loss = ...
    value_loss = ...
    return policy_loss, value_loss

In [ ]:
_lp = torch.tensor([-0.5, -1.0, -2.0], requires_grad=True)
_V = torch.tensor([1.0, 1.0, 2.0], requires_grad=True)
_G = torch.tensor([3.0, 2.0, 1.0])
_losses = baseline_losses(_lp, _G, _V)
must_fill(baseline_losses=_losses)
_policy_loss, _value_loss = _losses
# psi = (2, 1, -1): policy loss = -(2·-0.5 + 1·-1 + -1·-2) = 0; value loss = 4 + 1 + 1 = 6
assert torch.isclose(_policy_loss, torch.tensor(0.0)), f"policy loss: got {_policy_loss.item():.3f}, expected 0"
assert torch.isclose(_value_loss, torch.tensor(6.0)), f"value loss: got {_value_loss.item():.3f}, expected 6"
_policy_loss.backward()
assert _V.grad is None, "the policy loss reaches the value network: detach V in psi"
_value_loss.backward()
assert torch.allclose(_V.grad, -2 * (_G - _V.detach())), "the value loss must train V"
print("  the losses of Algorithm 2 are correct ✓")

## 3.2 The loop

The same loop as Algorithm 1, with two changes: the multiplier is $G_t - V_\phi(s_t)$, and after the policy's step
the value network takes a step on this batch's returns. Each network has its own optimiser, with step sizes
$\alpha = 2\cdot10^{-3}$ for the policy and $\beta = 10^{-3}$ for the value network. One more detail: the value
network learns better on numbers of a modest size, while LunarLander's returns reach the hundreds, so the rewards are divided by
10 before the returns are computed. That rescales the multiplier and changes nothing else.

In [ ]:
def reinforce_baseline(seed: int, total_steps: int, batch_size: int = 8, lr_policy: float = 2e-3,
                       lr_value: float = 1e-3, reward_scale: float = 0.1, gamma: float = 0.99,
                       eval_every: int = 25_000) -> dict:
    "Algorithm 2: REINFORCE with a learned baseline V_phi(s); returns training returns, evaluations and weights."
    torch.manual_seed(seed)
    env = make_env()
    env.reset(seed=seed)          # seed the environment once; every later reset continues its random stream
    policy = Policy(env.observation_space.shape[0], env.action_space)
    value_net = util.mlp(env.observation_space.shape[0], 1)
    opt_policy = torch.optim.Adam(policy.parameters(), lr=lr_policy)      # alpha
    opt_value = torch.optim.Adam(value_net.parameters(), lr=lr_value)     # beta
    returns, steps_at, evals, steps, next_eval = [], [], [], 0, eval_every
    while steps < total_steps:
        policy_loss, value_loss = 0.0, 0.0
        for _ in range(batch_size):
            states, log_probs, rewards = run_episode(env, policy)
            G = torch.as_tensor(discounted_returns([reward_scale * r for r in rewards], gamma), dtype=torch.float32)
            p_loss, v_loss = baseline_losses(log_probs, G, value_net(states).squeeze(-1))
            policy_loss = policy_loss + p_loss / batch_size
            value_loss = value_loss + v_loss / batch_size
            steps += len(rewards)
            returns.append(sum(rewards))
            steps_at.append(steps)
        opt_policy.zero_grad()
        policy_loss.backward()
        opt_policy.step()                                               # the policy's step
        opt_value.zero_grad()
        value_loss.backward()
        opt_value.step()                                                # then the value network's
        while steps >= next_eval:
            evals.append((next_eval, evaluate(sampled(policy), seed=10**6 + next_eval)))
            next_eval += eval_every
    return {"seed": seed, "returns": returns, "steps_at": steps_at, "evals": evals, "weights": policy.state_dict()}

## 3.3 Training and results

Same seeds and the same budget of environment steps as Part 2.

In [ ]:
t0 = time.perf_counter()
baseline_runs = util.parallel_map(lambda seed: reinforce_baseline(seed, N_STEPS, gamma=GAMMA,
                                                                  eval_every=EVAL_EVERY), SEEDS)
print(f"{len(SEEDS)} seeds × {N_STEPS:,} steps in {time.perf_counter() - t0:.0f} s")
plot_methods({"REINFORCE": (reinforce_runs, AMBER), "with a baseline": (baseline_runs, GREEN)},
             "REINFORCE with and without a baseline, same seeds")
show_policies(baseline_runs, "baseline.mp4")
RESULTS["REINFORCE with a baseline"] = float(np.mean([final_score(r) for r in baseline_runs]))

**What to look for.** The baseline should lift both curves well above REINFORCE's, seed by seed. In the videos,
compare these landers with REINFORCE's alone: the policies trained with a baseline are much more careful, with a
slower, steadier descent. What a baseline cannot fix: $G_t$ still waits for the end of the episode, and it carries the luck of every later step — here up to
a thousand of them. Part 4 takes that on, and there the value network, renamed the critic, starts to matter in a
second, more dangerous way: its errors no longer only add noise.

## ⭐ Bonus 2 — The best constant baseline

*Nothing later depends on this cell.*

A baseline need not depend on the state at all. One can derive the constant $b$ that minimises the variance of
the gradient estimate:

$$b^\star = \frac{\mathbb{E}\bigl[\lVert f_t\rVert^2\, G_t\bigr]}{\mathbb{E}\bigl[\lVert f_t\rVert^2\bigr]},
\qquad f_t = \nabla_w \log\pi_w(a_t \mid s_t),$$

a weighted average of the returns, weighted by the squared length of each step's score. Two things to try, each
with its cell below:

**(a)** Estimate $b^\star$ on 16 episodes of the untrained policy and compare it with the plain average of the returns.
   The first cell gets the score of each step for you; fill its two 🎯 lines. How different are they, and why?

**(b)** Train with $\Psi_t = G_t - \bar G$, the batch's mean return as a constant baseline. The second cell does it and
   plots it beside REINFORCE and the learned baseline, with the same budget (about 7 minutes). Where does it land, and why?

In [ ]:
# ⭐ Bonus 2 (a) — the best constant baseline b*, estimated on the untrained policy (about 20 seconds).
RUN_BONUS = False
if RUN_BONUS:
    torch.manual_seed(SEEDS[0])
    probe_b = Policy(OBS_DIM, ACTION_SPACE)
    env_b = make_env()
    env_b.reset(seed=0)
    sq_norms, step_returns = [], []                 # ||f_t||² and G_t, for every step of every episode
    for _ in range(16):
        _, log_probs, rewards = run_episode(env_b, probe_b)
        G = discounted_returns(rewards, GAMMA)
        for t in range(len(rewards)):
            # the score f_t = ∇_w log π_w(a_t | s_t): the gradient of this one step's log-probability, layer by layer
            f_t = torch.autograd.grad(log_probs[t], list(probe_b.parameters()), retain_graph=True)
            sq_norms.append(...)      # 🎯 ||f_t||², summed over every weight
            step_returns.append(G[t])
    must_fill(_where="this cell", squared_norm=... if Ellipsis in sq_norms else sq_norms)
    sq_norms, step_returns = np.array(sq_norms, dtype=float), np.array(step_returns)
    b_star = ...         # 🎯 the formula above, with the two arrays
    must_fill(_where="this cell", b_star=b_star)
    print(f"b* = {b_star:7.2f}   the plain average of G_t = {step_returns.mean():7.2f}   over {len(step_returns)} steps")
    print(f"||f_t||² ranges from {sq_norms.min():.2f} to {sq_norms.max():.2f}: the steps with long scores weigh more in b*")

In [ ]:
# ⭐ Bonus 2 (b) — REINFORCE with the batch mean as a constant baseline (about 7 minutes; same budget as the main runs).
RUN_BONUS = False
if RUN_BONUS:
    def reinforce_mean_baseline(seed, total_steps, batch_size=8, lr=1e-3, gamma=0.99, eval_every=25_000):
        torch.manual_seed(seed)
        env = make_env()
        env.reset(seed=seed)          # seed the environment once; every later reset continues its random stream
        policy = Policy(OBS_DIM, ACTION_SPACE)
        opt = torch.optim.Adam(policy.parameters(), lr=lr)
        returns, steps_at, evals, steps, next_eval = [], [], [], 0, eval_every
        while steps < total_steps:
            episodes = []
            for _ in range(batch_size):
                _, log_probs, rewards = run_episode(env, policy)
                episodes.append((log_probs, torch.as_tensor(discounted_returns(rewards, gamma), dtype=torch.float32)))
                steps += len(rewards)
                returns.append(sum(rewards))
                steps_at.append(steps)
            b = torch.cat([G for _, G in episodes]).mean()                 # the batch's mean return, one number
            loss = sum(reinforce_loss(lp, G - b) for lp, G in episodes) / batch_size
            opt.zero_grad()
            loss.backward()
            opt.step()
            while steps >= next_eval:
                evals.append((next_eval, evaluate(sampled(policy), seed=10**6 + next_eval)))
                next_eval += eval_every
        return {"seed": seed, "returns": returns, "steps_at": steps_at, "evals": evals, "weights": policy.state_dict()}

    experiment(reinforce_mean_baseline, {"constant baseline G-bar": {}}, "A constant baseline between the two",
               steps=N_STEPS, reference={"REINFORCE": reinforce_runs, "learned baseline V(s)": baseline_runs})
TIMINGS["3 REINFORCE with a baseline"] = time.perf_counter() - PART_T0

# Part 4 — Actor-critic (Algorithm 3)

**The problem, seen in Part 3.** REINFORCE judges an action by everything that happened after it, and only once
the episode is over — here often hundreds of steps later. A good action followed by bad luck is punished; and
nothing is learned until the end.

**Two new names.** The value network $V_\phi$ of Part 3 is now called the **critic**, because it judges actions,
and the policy network is called the **actor**.

**The idea.** Judge each action one step later. Take one real step, get the reward $r_t$, and let the critic
estimate the rest. The **Bellman equation** says that, on average, the value of a state is one reward plus the
value of the next state:

$$V(s_t) = \mathbb{E}\bigl[\,r_t + \gamma\,V(s_{t+1}) \bigm| s_t\,\bigr].$$

So $y_t = r_t + \gamma V_\phi(s_{t+1})$ is a fair target for $V_\phi(s_t)$, and the gap between them, the
**one-step error**

$$\delta_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t),$$

says whether the action left things better or worse than the critic expected. It becomes the multiplier. If the
episode ended ($s_{t+1}$ is terminal), there is no future: $y_t = r_t$.

**The update, after every step:** the actor takes a step on $-\delta_t \log\pi_w(a_t\mid s_t)$ and the critic a
step on $\tfrac12\delta_t^2$ (with $y_t$ held fixed). Training the critic on its own estimate one step later is
called **bootstrapping**.

**What this costs.** In Part 3 the value network only *compared*; here it also *stands in for the future*. A wrong critic
no longer just adds noise: it points the actor the wrong way. At the start the critic knows nothing, so $\delta_t$
is mostly noise and the actor is pushed in random directions. Three choices keep that under control:
- **Parallel environments.** As the lecture notes, in practice many copies of the environment run side by side
  (Mnih et al. 2016), and each step averages the losses over all of them. We use 16; 4.4 tests the number.
- **An actor slower than the critic:** $\alpha = 3\cdot10^{-4}$ against $\beta = 10^{-2}$, so that the critic is
  roughly right before the actor relies on it.
- **Rewards divided by 10**, as in Part 3, so that the critic's targets stay of a modest size.

Many actor-critic codes add a fourth, an **entropy bonus** that rewards the policy for staying a little random (PPO
in Part 6 has one). The loop below has it as an option, off by default.

In [ ]:
PART_T0 = time.perf_counter()

## 4.1 One step of Algorithm 3

Four lines, as on the slide "Algorithm 3 in PyTorch", for all environments at once (every argument holds one value
per environment):
1. the target $y = r + \gamma V_\phi(s_{t+1})(1 - \text{terminated})$, computed **without gradient**;
2. the one-step error $\delta = y - V_\phi(s_t)$;
3. the actor's loss $-\delta \log\pi_w(a_t \mid s_t)$, with $\delta$ **detached**, averaged over the environments;
4. the critic's loss $\tfrac12\delta^2$, averaged over the environments.

In [ ]:
# 🎯 YOUR TURN — Exercise 4: one step of Algorithm 3, for all environments at once.
#
# 🎯 y = r + γ V_next (1 - terminated), without gradient;  delta = y - V_s;
#    actor loss = mean of -delta·log π with delta detached;  critic loss = mean of ½ delta².

def actor_critic_losses(log_prob: torch.Tensor, V_s: torch.Tensor, V_next: torch.Tensor,
                        r: torch.Tensor, terminated: torch.Tensor, gamma: float):
    "Return (actor loss, critic loss) for one step of every environment, with the multiplier delta = y - V_phi(s_t)."
    with torch.no_grad():
        y = ...
    delta = ...
    actor_loss = ...
    critic_loss = ...
    return actor_loss, critic_loss

In [ ]:
# Two environments: the first goes on (y = 1 + 0.5·4 = 3, δ = 3 - 2 = 1), the second's episode ended (y = 1, δ = -1)
_lp = torch.tensor([-0.5, -1.0], requires_grad=True)
_Vs = torch.tensor([2.0, 2.0], requires_grad=True)
_Vn = torch.tensor([4.0, 4.0], requires_grad=True)
_losses = actor_critic_losses(_lp, _Vs, _Vn, r=torch.tensor([1.0, 1.0]), terminated=torch.tensor([0.0, 1.0]), gamma=0.5)
must_fill(actor_critic_losses=_losses)
_actor, _critic = _losses
assert torch.isclose(_actor, torch.tensor(-0.25)), f"actor: got {_actor.item():.3f}, expected mean(-1·-0.5, 1·-1) = -0.25"
assert torch.isclose(_critic, torch.tensor(0.5)), f"critic: got {_critic.item():.3f}, expected mean(½·1, ½·1) = 0.5"
(_actor + _critic).backward()
assert _Vn.grad is None, "the target y must be held fixed: compute it without gradient"
assert torch.allclose(_Vs.grad, torch.tensor([-0.5, 0.5])), "the critic's gradient should be -δ / 2 per environment"
print("  one step of Algorithm 3 is correct ✓")

## 4.2 The loop

`gym.vector.SyncVectorEnv` steps 16 copies of LunarLander together and returns arrays with one row per copy. When a
copy's episode ends, it starts a new one on its own (`AutoresetMode.SAME_STEP`): the state it returns is already
the new episode's first, and the real last state is in `info["final_obs"]` — that is the $s_{t+1}$ we need for the
target. Nothing waits for the end of an episode: after every step of the 16 copies, the actor and the critic each
take one gradient step, from one optimiser with two step sizes. The entropy bonus is in the loss but, by default,
weighted by zero.

In [ ]:
def actor_critic(seed: int, total_steps: int, n_envs: int = 16, lr_actor: float = 3e-4,
                 lr_critic: float = 1e-2, ent_coef: float = 0.0, reward_scale: float = 0.1,
                 gamma: float = 0.99, eval_every: int = 25_000) -> dict:
    "Algorithm 3 on n_envs parallel copies: one update of actor and critic after every step."
    torch.manual_seed(seed)
    envs = gym.vector.SyncVectorEnv([make_env] * n_envs, autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
    actor = Policy(envs.single_observation_space.shape[0], envs.single_action_space)
    critic = util.mlp(envs.single_observation_space.shape[0], 1)
    opt = torch.optim.Adam([{"params": actor.parameters(), "lr": lr_actor},      # alpha
                            {"params": critic.parameters(), "lr": lr_critic}])   # beta
    obs, _ = envs.reset(seed=seed)
    running = np.zeros(n_envs)
    returns, steps_at, evals, steps, next_eval = [], [], [], 0, eval_every
    while steps < total_steps:
        dist = actor(torch.as_tensor(obs, dtype=torch.float32))
        a = dist.sample()
        next_obs, r, terminated, truncated, info = envs.step(a.numpy())
        s_next = next_obs.copy()
        for i in np.flatnonzero(terminated | truncated):
            s_next[i] = info["final_obs"][i]                  # the real last state, not the new episode's first
        V = critic(torch.as_tensor(np.concatenate([obs, s_next]), dtype=torch.float32)).squeeze(-1)
        actor_loss, critic_loss = actor_critic_losses(
            dist.log_prob(a), V[:n_envs], V[n_envs:], torch.as_tensor(reward_scale * r, dtype=torch.float32),
            torch.as_tensor(terminated, dtype=torch.float32), gamma)
        entropy_bonus = ent_coef * dist.entropy().mean()      # optional: rewards a policy that stays random
        opt.zero_grad()
        (actor_loss + critic_loss - entropy_bonus).backward()
        opt.step()
        steps += n_envs
        running += r
        for i in np.flatnonzero(terminated | truncated):        # bookkeeping of finished episodes
            returns.append(running[i])
            steps_at.append(steps)
            running[i] = 0.0
        while steps >= next_eval:
            evals.append((next_eval, evaluate(sampled(actor), seed=10**6 + next_eval)))
            next_eval += eval_every
        obs = next_obs
    envs.close()
    return {"seed": seed, "returns": returns, "steps_at": steps_at, "evals": evals, "weights": actor.state_dict()}

## 4.3 Training and the three methods compared

Same seeds and the same budget of environment steps as Parts 2 and 3.

In [ ]:
t0 = time.perf_counter()
ac_runs = util.parallel_map(lambda seed: actor_critic(seed, N_STEPS, gamma=GAMMA, eval_every=EVAL_EVERY), SEEDS)
print(f"{len(SEEDS)} seeds × {N_STEPS:,} steps in {time.perf_counter() - t0:.0f} s")
plot_methods({"REINFORCE": (reinforce_runs, AMBER), "with a baseline": (baseline_runs, GREEN),
              "actor-critic": (ac_runs, PURPLE2)}, "The three multipliers on LunarLander, same seeds")
show_policies(ac_runs, "actor_critic.mp4")
RESULTS["Actor-critic"] = float(np.mean([final_score(r) for r in ac_runs]))

| Method | Multiplier $\Psi_t$ | Compares with what usually happens | Waits for | Unbiased |
|---|---|---|---|---|
| REINFORCE (Part 2) | $G_t$ | no | the end of the episode | yes |
| With a baseline (Part 3) | $G_t - V_\phi(s_t)$ | yes | the end of the episode | yes |
| Actor-critic (Part 4) | $\delta_t$ | yes | one step | only if the critic is right |

With exact values, all three give the same gradient on average; they differ in how noisy they are and in how long
they wait. The table below gives each seed's final score, method by method.

In [ ]:
names = ["REINFORCE", "with baseline", "actor-critic"]
all_runs = (reinforce_runs, baseline_runs, ac_runs)
print(f"{'':>8}" + "".join(f"{n:>16}" for n in names) + "   (final score: mean of the last three evaluations)")
for i, seed in enumerate(SEEDS):
    print(f"{seed:>8}" + "".join(f"{final_score(runs[i]):>16.0f}" for runs in all_runs))


## 4.4 🔬 Experiment: parallel environments

Of the three practical choices, the one you will meet again in every actor-critic code is how many environments run
side by side.

**Questions to answer from the plot.** (1) With one environment, each $\delta_t$ is one step of one episode: does
the actor still learn, and how steadily? (2) More environments give a steadier gradient, but, at a fixed budget of
environment steps, fewer updates. Where does the gain stop, and what happens at 64? (Keep in mind that a setting can look harmless at
this budget and still matter at a longer one — Part 4's main run uses more steps than the experiment does.)

In [ ]:
envs_runs = experiment(actor_critic, {"1 environment": dict(n_envs=1), "4 environments": dict(n_envs=4),
                                      "16 environments (ours)": dict(n_envs=16), "64 environments": dict(n_envs=64)},
                       "Actor-critic: environments stepped in parallel, same budget of environment steps")

**Reading the experiment.** The number of parallel environments is a knob on the *noise of one update*: one
environment gives the critic and the actor a fresh error at every step, and too many leave too few updates in the
budget. Both ends meet the same danger, a critic that is wrong at the start and an actor that believes it. PPO, in
Part 6, keeps the parallel environments and adds a guard of its own.

## ⭐ Bonus 3 — Is the critic consistent?

*Nothing later depends on this cell.*

The Bellman equation says that, for the policy's own states, the one-step error $\delta_t$ is zero **on average**.
That gives a test of the critic that needs no true values. The cell below trains Part 4's actor-critic once more,
keeping both networks after 50,000 steps and at the end, and plays 10 episodes with each sampled actor. Fill its one
🎯 line, which computes $\delta_t$ (with the rewards divided by 10, as in training); the cell then prints the mean and
spread of the errors and plots where in an episode they fall (about 3 minutes). Is the late critic consistent, and is the early one? Why is a
mean of zero necessary but not enough for a good critic?

In [ ]:
# ⭐ Bonus 3 — a starting point: is the critic consistent? Fill the one 🎯 line (training takes about 3 minutes).
RUN_BONUS = False
if RUN_BONUS:
    import copy

    def actor_critic_snapshots(seed, total_steps, snapshot_at=50_000, n_envs=16, lr_actor=3e-4, lr_critic=1e-2,
                               reward_scale=0.1):
        "Part 4's loop (4.2), returning copies of (actor, critic) after `snapshot_at` steps and at the end."
        torch.manual_seed(seed)
        envs = gym.vector.SyncVectorEnv([make_env] * n_envs, autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
        actor, critic = Policy(OBS_DIM, ACTION_SPACE), util.mlp(OBS_DIM, 1)
        opt = torch.optim.Adam([{"params": actor.parameters(), "lr": lr_actor},
                                {"params": critic.parameters(), "lr": lr_critic}])
        obs, _ = envs.reset(seed=seed)
        snapshots = {}
        for step in range(0, total_steps, n_envs):
            if step >= snapshot_at and "early" not in snapshots:
                snapshots["early"] = (copy.deepcopy(actor), copy.deepcopy(critic))
            dist = actor(torch.as_tensor(obs, dtype=torch.float32))
            a = dist.sample()
            next_obs, r, terminated, truncated, info = envs.step(a.numpy())
            s_next = next_obs.copy()
            for i in np.flatnonzero(terminated | truncated):
                s_next[i] = info["final_obs"][i]
            V = critic(torch.as_tensor(np.concatenate([obs, s_next]), dtype=torch.float32)).squeeze(-1)
            actor_loss, critic_loss = actor_critic_losses(
                dist.log_prob(a), V[:n_envs], V[n_envs:], torch.as_tensor(reward_scale * r, dtype=torch.float32),
                torch.as_tensor(terminated, dtype=torch.float32), GAMMA)
            opt.zero_grad()
            (actor_loss + critic_loss).backward()
            opt.step()
            obs = next_obs
        envs.close()
        snapshots["late"] = (actor, critic)
        return snapshots

    def deltas_of(actor, critic, n_episodes=10, seed=3000, reward_scale=0.1):
        "The one-step errors δ_t of n episodes played with the sampled actor: one array per episode."
        env, episodes = make_env(), []
        for i in range(n_episodes):
            obs, _ = env.reset(seed=seed + i)
            deltas = []
            while True:
                with torch.no_grad():
                    a = actor(torch.as_tensor(obs, dtype=torch.float32)).sample()
                obs2, r, terminated, truncated, _ = env.step(to_env_action(a))
                with torch.no_grad():
                    V_s = critic(torch.as_tensor(obs, dtype=torch.float32)).item()
                    V_next = critic(torch.as_tensor(obs2, dtype=torch.float32)).item()
                # 🎯 δ_t = reward_scale · r + γ V(s_{t+1}) − V(s_t), with no future after a real end (terminated)
                delta = ...
                deltas.append(delta)
                obs = obs2
                if terminated or truncated:
                    break
            episodes.append(np.array(deltas, dtype=float))
        return episodes

    snaps = actor_critic_snapshots(SEEDS[0], N_STEPS)
    try:
        by_stage = {name: deltas_of(*nets) for name, nets in snaps.items()}
    except TypeError:                             # `...` is still in deltas_of
        by_stage = ...
    must_fill(_where="this cell, in deltas_of", delta=by_stage)
    fig, ax = plt.subplots(figsize=(8, 3.2))
    for (name, episodes), color in zip(by_stage.items(), [AMBER, PURPLE2]):
        allx = np.concatenate(episodes)
        print(f"{name:5s} critic: mean δ {allx.mean():+.3f}   spread {allx.std():.3f}   over {len(allx)} steps")
        longest = max(len(e) for e in episodes)
        mean_abs = [np.mean([abs(e[t]) for e in episodes if len(e) > t]) for t in range(longest)]
        ax.plot(util.smooth(np.array(mean_abs), 10), color=color, label=f"{name} critic")
    ax.set_xlabel("step of the episode")
    ax.set_ylabel("mean |δ_t| (10-step average)")
    ax.set_title("Where the critic's one-step errors fall")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
TIMINGS["4 Actor-critic"] = time.perf_counter() - PART_T0

# Part 5 — Generalised advantage estimation (GAE)

*Parts 5 and 6 are given code, explained step by step: read, run, and look at the results.*

**The problem, seen in Parts 3 and 4.** Part 4's multiplier $\delta_t$ uses one real reward and lets the critic
guess the rest: little noise, but wrong whenever the critic is wrong. Part 3's $G_t - V_\phi(s_t)$ uses every real
reward: never biased by the critic, but noisy. We had to choose between the two. GAE makes the choice a dial.

**The idea.** Instead of one real reward before the critic takes over, use two, three, or more, and take a
weighted average of all these choices. Worked out, that average is a sum of future one-step errors whose weights
shrink by a factor $\gamma\lambda$ per step:

$$\hat A_t = \delta_t + (\gamma\lambda)\,\delta_{t+1} + (\gamma\lambda)^2\,\delta_{t+2} + \dots
= \delta_t + \gamma\lambda\,\hat A_{t+1}.$$

The second form is computed in one pass from the last step back to the first.

| $\lambda$ | Multiplier | Noise from | If the critic is wrong |
|---|---|---|---|
| 0 | $\delta_t$ (Part 4) | one step | biased |
| between | $\hat A_t$ | later steps, fading | a little biased |
| 1 | $G_t - V_\phi(s_t)$ (Part 3) | every later step | unbiased |

PPO uses $\gamma = 0.99$ and $\lambda = 0.95$ (Schulman et al. 2017). Part 6's experiment turns the dial.

## 5.1 The pass, in code

`compute_gae` is the slide "The same pass in PyTorch", for whole rollouts of several environments at once (arrays
of shape `(T, num_envs)`). Two details:
- `values` has one more entry than `rewards`: the critic's value of the state after the last step, so that a rollout
  that stops in the middle of an episode can be finished by the critic;
- `dones[t] = 1` marks the last step of an episode. There the sum stops: the next state belongs to a *new* episode.

In [ ]:
PART_T0 = time.perf_counter()

def compute_gae(rewards: np.ndarray, values: np.ndarray, dones: np.ndarray,
                gamma: float, lam: float) -> np.ndarray:
    '''Generalised advantage estimates for a rollout.

    rewards, dones: shape (T, ...); dones[t] = 1 if the episode ended at that step.
    values:         shape (T+1, ...); values[T] is the critic's value of the state after the last step.
    Returns the advantages, shape (T, ...). The critic's targets are advantages + values[:-1].
    '''
    T = len(rewards)
    A = np.zeros(np.shape(rewards), dtype=np.float64)
    A_next = 0.0
    for t in reversed(range(T)):                      # from the last step back to the first
        not_done = 1.0 - dones[t]
        delta = rewards[t] + gamma * values[t + 1] * not_done - values[t]
        A[t] = delta + gamma * lam * not_done * A_next
        A_next = A[t]
    return A

## 5.2 The lecture's example

From the slide "One right-to-left pass": $\gamma = 1$, $\lambda = \tfrac12$, a critic that grows more hopeful by 0.2
a step, then a lost episode. The one-step errors are $+0.2, +0.2, +0.2, -1.8$. From the right: $-1.8$, then
$0.2 + \tfrac12(-1.8) = -0.7$, then $-0.15$, then $+0.125$. The bad news of the last step reaches back, halved at
every step.

In [ ]:
A_lecture = compute_gae(rewards=np.array([0.0, 0.0, 0.0, -1.0]),
                        values=np.array([0.2, 0.4, 0.6, 0.8, 0.0]),    # the value after the end is 0
                        dones=np.array([0.0, 0.0, 0.0, 1.0]), gamma=1.0, lam=0.5)
print("Â_1 ... Â_4 =", A_lecture)
np.testing.assert_allclose(A_lecture, [0.125, -0.15, -0.7, -1.8])

## 5.3 The two ends of the dial, across an episode boundary

Four steps, $\gamma = 0.5$: a first episode that ends at $t = 2$, and a second one still running when the rollout
stops, finished by the critic with $V_\phi(s_5) = 2.5$. Computed by hand:

| $t$ | $r_t$ | $V_\phi(s_t)$ | done | $\delta_t$ ($\lambda=0$) | $G_t - V_\phi(s_t)$ ($\lambda=1$) | $\lambda = 0.5$ |
|---|---|---|---|---|---|---|
| 1 | 1 | 0.5 | 0 | $1 + 0.5\cdot1.0 - 0.5 = 1.0$ | $2 - 0.5 = 1.5$ | $1.0 + 0.25\cdot1.0 = 1.25$ |
| 2 | 2 | 1.0 | **1** | $2 + 0 - 1.0 = 1.0$ | $2 - 1.0 = 1.0$ | 1.0 |
| 3 | 3 | 1.5 | 0 | $3 + 0.5\cdot2.0 - 1.5 = 2.5$ | $5.625 - 1.5 = 4.125$ | $2.5 + 0.25\cdot3.25 = 3.3125$ |
| 4 | 4 | 2.0 | 0 | $4 + 0.5\cdot2.5 - 2.0 = 3.25$ | $5.25 - 2.0 = 3.25$ | 3.25 |

In [ ]:
rewards = np.array([1.0, 2.0, 3.0, 4.0])
values  = np.array([0.5, 1.0, 1.5, 2.0, 2.5])          # 5 entries: the last finishes the open episode
dones   = np.array([0.0, 1.0, 0.0, 0.0])
expected = {0.0: [1.0, 1.0, 2.5, 3.25], 0.5: [1.25, 1.0, 3.3125, 3.25], 1.0: [1.5, 1.0, 4.125, 3.25]}
for lam, by_hand in expected.items():
    A = compute_gae(rewards, values, dones, 0.5, lam)
    np.testing.assert_allclose(A, by_hand)
    print(f"λ = {lam}:  Â = {A}   (matches the table)")
print("without the done flag, δ_2 would be", 2 + 0.5 * values[2] - values[1], "instead of 1.0: "
      "the first episode would borrow the value of the second")
TIMINGS["5 GAE"] = time.perf_counter() - PART_T0

> **⚠️ Episode boundaries.** In a rollout of parallel environments, one column holds the end of one episode and the
> start of the next. Two flags matter:
> - **terminated** (the lander came to rest or crashed): the future is worth 0, so the sum stops.
> - **truncated** (the 1,000-step limit): the episode is cut, but the state still had a future. The sum stops too,
>   but the last reward gets $\gamma V_\phi(s_\text{final})$ added (6.2). Gymnasium gives the real last state in
>   `info["final_obs"]`.

## ⭐ Bonus 4 — The rungs of the ladder

*Nothing later depends on this cell.*

GAE was introduced as a weighted average of $n$-step estimates: $n$ real rewards, then the critic. Make
that average concrete: `n_step_advantage(rewards, values, dones, gamma, n)`
returns $r_t + \gamma r_{t+1} + \dots + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n}) - V(s_t)$ for every $t$,
stopping early at an episode's end. The cell below has the loop: fill its two 🎯 lines, and it checks on the table of 5.3 that $n = 1$ gives $\delta_t$ and a large $n$ gives
$G_t - V(s_t)$, and that $\hat A_t$ at $\lambda = 0.5$ equals $(1-\lambda)\sum_{n\ge1} \lambda^{n-1}\,(\text{the
}n\text{-step advantage})$, summed as far as the episode goes (at a finite horizon the last rung takes the remaining weight; the cell does that for you).

In [ ]:
# ⭐ Bonus 4 — a starting point: the n-step rungs, and GAE as their weighted average. Fill the two 🎯 lines.
RUN_BONUS = False
if RUN_BONUS:
    def n_step_advantage(rewards, values, dones, gamma, n):
        "r_t + γ r_{t+1} + ... + γ^(k-1) r_{t+k-1} + γ^k V(s_{t+k}) − V(s_t), with k = n, or fewer at an episode's end."
        T, A = len(rewards), np.zeros(len(rewards))
        for t in range(T):
            total, k, ended = 0.0, 0, False
            while k < n and t + k < T and not ended:
                total += ...            # 🎯 the discounted reward of step t + k
                ended = bool(dones[t + k])
                k += 1
            if not ended:
                total += ...             # 🎯 the critic finishes the sum: γ^k V(s_{t+k})
            A[t] = total - values[t]
        return A

    rewards, values, dones = np.array([1.0, 2.0, 3.0, 4.0]), np.array([0.5, 1.0, 1.5, 2.0, 2.5]), np.array([0.0, 1.0, 0.0, 0.0])
    try:
        rung_1 = n_step_advantage(rewards, values, dones, 0.5, 1)
    except TypeError:                             # `...` is still in the loop
        rung_1 = ...
    must_fill(_where="this cell, in n_step_advantage", rung=rung_1)
    rung_all = n_step_advantage(rewards, values, dones, 0.5, 100)
    print("n = 1:   ", rung_1, "  5.3's λ = 0 column, the δ_t:      [1.0, 1.0, 2.5, 3.25]")
    print("n = 100: ", rung_all, "  5.3's λ = 1 column, G_t − V(s_t): [1.5, 1.0, 4.125, 3.25]")
    lam, T = 0.5, len(rewards)
    rungs = {n: n_step_advantage(rewards, values, dones, 0.5, n) for n in range(1, T + 1)}
    average = np.zeros(T)
    for t in range(T):
        N = next(k for k in range(1, T - t + 1) if dones[t + k - 1] or t + k == T)   # rungs available from step t
        average[t] = sum((1 - lam) * lam ** (n - 1) * rungs[n][t] for n in range(1, N)) + lam ** (N - 1) * rungs[N][t]
    print("weights (1−λ)λ^(n−1), the last rung taking the rest:", average,
          "  compute_gae at λ = 0.5:", compute_gae(rewards, values, dones, 0.5, lam))
    np.testing.assert_allclose(rung_1, [1.0, 1.0, 2.5, 3.25])
    np.testing.assert_allclose(rung_all, [1.5, 1.0, 4.125, 3.25])
    np.testing.assert_allclose(average, compute_gae(rewards, values, dones, 0.5, lam))
    print("  the rungs are correct, and GAE is their weighted average ✓")

# Part 6 — PPO (Algorithm 4)

**The problem, seen in every part so far.** Each method takes **one** gradient step on a batch and throws the batch
away: after the step, the batch comes from a policy that no longer exists. That wastes data (collecting it means
playing the simulator step by step, the slow part of training), and one step that is too large can ruin the policy, from which the next
batch is collected.

**The idea.** Take $K$ steps on the same batch, but correct for the policy moving away from the one that collected
it, $\pi_{w_\text{old}}$. The **probability ratio**

$$\rho_t(w) = \frac{\pi_w(a_t\mid s_t)}{\pi_{w_\text{old}}(a_t\mid s_t)}$$

says how much more likely the action is now than when it was collected. PPO maximises

$$L(w) = \operatorname{average}_t\Bigl[\min\bigl(\rho_t\,\Psi_t,\;
\operatorname{clip}(\rho_t, 1-\epsilon, 1+\epsilon)\,\Psi_t\bigr)\Bigr],\qquad \epsilon = 0.2,$$

with $\Psi_t = \hat A_t$ from GAE. On a fresh batch every $\rho_t = 1$ and the gradient of $L$ is the policy gradient.
The clip stops rewarding a change of an action's probability by more than about 20% within one batch.

PPO also keeps everything the previous parts built: parallel environments, a critic and GAE, and adds a small
entropy bonus of its own.

In [ ]:
PART_T0 = time.perf_counter()
PPO_TOTAL_STEPS = N_STEPS
PPO_EVAL_EVERY = max(1, EVAL_EVERY // 2048)   # evaluate every ~25,000 environment steps (12 batches of 2,048)

## 6.1 The agent

The actor and the critic in one module, plus a **normaliser** of the states: each of the 8 numbers is shifted and
scaled by running statistics before it enters a network, since a position and an angular velocity have very
different sizes. `act` turns a raw state into an action; `deterministic=True` takes the most likely action
instead of a sample (Bonus 1 (c) shows the difference).

In [ ]:
class Agent(nn.Module):
    "Actor, critic and state normaliser in one module, so that evaluation uses the training statistics."

    def __init__(self, obs_space: gym.Space, action_space: gym.Space):
        super().__init__()
        obs_dim = obs_space.shape[0]
        self.actor = Policy(obs_dim, action_space)
        self.critic = util.mlp(obs_dim, 1, out_gain=1.0)
        self.obs_rms = util.RunningMeanStd(obs_space.shape)

    def normalise(self, obs: np.ndarray) -> torch.Tensor:
        o = (obs - self.obs_rms.mean) / np.sqrt(self.obs_rms.var + 1e-8)
        return torch.as_tensor(np.clip(o, -10, 10), dtype=torch.float32)

    def value(self, obs_t: torch.Tensor) -> torch.Tensor:
        return self.critic(obs_t).squeeze(-1)

    @torch.no_grad()
    def act(self, obs: np.ndarray, deterministic: bool = False):
        dist = self.actor(self.normalise(obs))
        return to_env_action(dist.mode if deterministic else dist.sample())

    def snapshot(self) -> dict:
        "Plain data from which `from_snapshot` rebuilds the agent (to send it between processes)."
        return {"weights": self.state_dict(), "mean": self.obs_rms.mean, "var": self.obs_rms.var}


def from_snapshot(snap: dict) -> Agent:
    env = make_env()
    agent = Agent(env.observation_space, env.action_space)
    agent.load_state_dict(snap["weights"])
    agent.obs_rms.mean, agent.obs_rms.var = snap["mean"], snap["var"]
    return agent

## 6.2 Finishing a truncated episode

When an episode hits the 1,000-step limit with the lander still flying, it is cut, not over. We add
$\gamma V_\phi(s_\text{final})$ to its last reward and then treat the step as an ordinary episode end. If it crashed
or came to rest on that very step, that ending wins: nothing is added.

In [ ]:
def bootstrap_truncated(rewards: np.ndarray, terminated: np.ndarray, truncated: np.ndarray,
                        final_values: np.ndarray, gamma: float) -> np.ndarray:
    "Rewards with gamma * V(final state) added where an episode hit the time limit without terminating."
    return np.where(truncated & ~terminated, rewards + gamma * final_values, rewards)

# Four environments: still running, fell, cut by the limit, fell exactly at the limit
r = bootstrap_truncated(np.ones(4), terminated=np.array([False, True, False, True]),
                        truncated=np.array([False, False, True, True]), final_values=np.full(4, 5.0), gamma=0.5)
print("rewards after the bootstrap:", r, "→ only the environment cut by the limit gets 0.5 · 5 added")

## 6.3 Collecting a rollout

`collect_rollout` steps all environments `n_steps` times and stores what the update needs: the normalised state,
the action, its log-probability under $\pi_{w_\text{old}}$, the critic's value, the reward and the done flag. On the
way it updates the state normaliser, records the return of every finished episode, divides the rewards by a running
estimate of the spread of the return (so the critic's targets keep a similar size), and finishes truncated episodes
with 6.2.

In [ ]:
def collect_rollout(envs: gym.vector.VectorEnv, agent: Agent, obs: np.ndarray, n_steps: int, gamma: float,
                    ret_rms: util.RunningMeanStd, ep: dict) -> tuple[dict, np.ndarray]:
    "Step all environments n_steps times; return a (T, N, ...) buffer and the next state."
    buf = {k: [] for k in ["obs", "act", "logp", "val", "rew", "done"]}
    for _ in range(n_steps):
        agent.obs_rms.update(obs)                              # the normaliser follows the data
        o = agent.normalise(obs)
        with torch.no_grad():
            dist = agent.actor(o)
            a = dist.sample()
            logp = dist.log_prob(a)
            v = agent.value(o)
        next_obs, r, term, trunc, info = envs.step(to_env_action(a, envs.single_action_space))
        done = term | trunc

        ep["ret"] += r                                         # the raw return of every finished episode
        for i in np.flatnonzero(done):
            ep["returns"].append(ep["ret"][i])
            ep["steps_at"].append(ep["global_step"] + i)
            ep["ret"][i] = 0.0
        ep["global_step"] += envs.num_envs

        ep["disc"] = ep["disc"] * gamma + r                    # rewards divided by the spread of the return
        ret_rms.update(ep["disc"])
        ep["disc"][done] = 0.0
        r_scaled = r / np.sqrt(ret_rms.var + 1e-8)

        final_values = np.zeros(envs.num_envs)                 # V_φ of the real last state, where truncated
        idx = np.flatnonzero(trunc & ~term)
        if len(idx):
            final = np.stack([info["final_obs"][i] for i in idx])
            with torch.no_grad():
                final_values[idx] = agent.value(agent.normalise(final)).numpy()
        r_scaled = bootstrap_truncated(r_scaled, term, trunc, final_values, gamma)

        for k, x in zip(buf, [o, a, logp, v, r_scaled, done]):
            buf[k].append(x)
        obs = next_obs                                         # already the new episode's first state where done
    return {k: torch.stack(v) if torch.is_tensor(v[0]) else np.stack(v) for k, v in buf.items()}, obs

## 6.4 The clipped objective

`ppo_policy_loss` returns $-L$ over a minibatch. What the clip does, for one sample:
- $\Psi_t > 0$ (a better action than expected): its probability may grow, but once $\rho_t > 1.2$ the clipped term
  is flat, its gradient is zero, and the sample stops pulling.
- $\Psi_t < 0$ (a worse action): its probability may shrink, but only until $\rho_t < 0.8$.
- The `min` takes the pessimistic term, so a ratio that moved the *wrong* way is never clipped: a mistake is always
  corrected.

The clip removes the reason to move far; it does not stop the policy from drifting. So practice logs the fraction
of clipped samples and an approximate KL divergence to the old policy.

In [ ]:
def ppo_policy_loss(new_logp: torch.Tensor, old_logp: torch.Tensor, psi: torch.Tensor, clip: float) -> torch.Tensor:
    "Minus L, PPO's clipped objective, over a minibatch."
    ratio = (new_logp - old_logp).exp()
    return -torch.min(ratio * psi, ratio.clamp(1 - clip, 1 + clip) * psi).mean()

The slide "One term of the objective, two signs", computed with `ppo_policy_loss` itself: one sample's term of $L$
as its ratio moves, for $\Psi_t = +1$ and $\Psi_t = -1$. Where the line is flat (orange), the gradient is zero.

In [ ]:
rho = torch.linspace(0.5, 1.5, 201)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, sign in zip(axes, [+1.0, -1.0]):
    new_logp = torch.log(rho).requires_grad_()
    terms = torch.stack([-ppo_policy_loss(lp.unsqueeze(0), torch.zeros(1), torch.tensor([sign]), 0.2)
                         for lp in new_logp])
    terms.sum().backward()
    flat = new_logp.grad.abs() < 1e-9
    ax.plot(rho, terms.detach(), color=PURPLE2, lw=2, label="one term of L")
    ax.plot(rho[flat], terms.detach()[flat], ".", color=AMBER, ms=3, label="zero gradient: clipped")
    for edge in [0.8, 1.0, 1.2]:
        ax.axvline(edge, ls=":", c="k", lw=0.8)
    ax.set_title(f"Ψ_t = {sign:+.0f}")
    ax.set_xlabel("ratio ρ_t")
    ax.legend(fontsize=8)
axes[0].set_ylabel("min(ρΨ, clip(ρ)Ψ)")
plt.show()

## 6.5 $K$ steps on one batch

`ppo_update` makes `epochs` passes over the batch, each in shuffled minibatches, one gradient step per minibatch.
Each step minimises

$$-L \;+\; c_v\,\tfrac12\bigl(V_\phi(s_t) - \hat R_t\bigr)^2 \;-\; c_e\,\mathcal H\bigl[\pi_w(\cdot\mid s_t)\bigr]:$$

the critic is fitted to the target $\hat R_t = \hat A_t + V_\phi(s_t)$, and a small bonus for the entropy
$\mathcal H$ (the randomness of the policy) keeps it exploring. Two common practices that the lecture does not show:
$\Psi_t$ is standardised within each minibatch, and the gradient's length is capped.

In [ ]:
def ppo_update(agent: Agent, opt: torch.optim.Optimizer, batch: dict, epochs: int, minibatch_size: int,
               clip: float, vf_coef: float, ent_coef: float, max_grad_norm: float) -> tuple[float, float]:
    "K gradient steps on one batch; return the mean approximate KL and the mean fraction of clipped samples."
    kls, clipfracs = [], []
    for epoch in range(epochs):
        for idx in torch.randperm(len(batch["obs"])).split(minibatch_size):
            dist = agent.actor(batch["obs"][idx])
            new_logp = dist.log_prob(batch["act"][idx])
            psi = batch["adv"][idx]
            psi = (psi - psi.mean()) / (psi.std() + 1e-8)
            policy_loss = ppo_policy_loss(new_logp, batch["logp"][idx], psi, clip)
            value_loss = 0.5 * ((agent.value(batch["obs"][idx]) - batch["ret"][idx]) ** 2).mean()
            loss = policy_loss + vf_coef * value_loss - ent_coef * dist.entropy().mean()
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), max_grad_norm)
            opt.step()
            with torch.no_grad():
                log_ratio = new_logp - batch["logp"][idx]
                kls.append((log_ratio.exp() - 1 - log_ratio).mean().item())
                clipfracs.append(((log_ratio.exp() - 1).abs() > clip).float().mean().item())
    return float(np.mean(kls)), float(np.mean(clipfracs))

## 6.6 The training loop

Each turn of the loop: lower the step size a little (it decays linearly to 0), collect a rollout (6.3), compute
$\Psi_t = \hat A_t$ with GAE (Part 5), flatten the `(T, N)` arrays into one batch, and take $K$ steps on it (6.5).
Every few batches, evaluate the policy. With 8 environments of 256 steps (2,048 samples), minibatches of 64 and 10
epochs, $K = 2048/64 \times 10 = 320$ steps per batch, the setting of Schulman et al. (2017) quoted in the lecture.

In [ ]:
def ppo(seed: int, total_steps: int, num_envs: int = 8, n_steps: int = 256, epochs: int = 10,
        minibatch_size: int = 64, gamma: float = 0.99, lam: float = 0.95, clip: float = 0.2, vf_coef: float = 0.5,
        ent_coef: float = 0.01, lr: float = 3e-4, max_grad_norm: float = 0.5, eval_every: int = 25_000) -> dict:
    "Algorithm 4 on num_envs copies of the environment; returns training returns, evaluations and the agent."
    torch.manual_seed(seed)
    np.random.seed(seed)
    envs = gym.vector.SyncVectorEnv([make_env] * num_envs, autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
    agent = Agent(envs.single_observation_space, envs.single_action_space)
    opt = torch.optim.Adam(agent.parameters(), lr=lr, eps=1e-5)
    ret_rms = util.RunningMeanStd()
    ep = dict(ret=np.zeros(num_envs), disc=np.zeros(num_envs), returns=[], steps_at=[], global_step=0)
    evals, kls, clipfracs, next_eval = [], [], [], eval_every
    obs, _ = envs.reset(seed=seed)
    n_updates = total_steps // (num_envs * n_steps)
    for update in range(n_updates):
        for g in opt.param_groups:
            g["lr"] = lr * (1 - update / n_updates)
        buf, obs = collect_rollout(envs, agent, obs, n_steps, gamma, ret_rms, ep)
        with torch.no_grad():                                  # V_φ of the state after the rollout
            last_v = agent.value(agent.normalise(obs)).numpy()
        values = np.concatenate([buf["val"].numpy(), last_v[None]])
        adv = compute_gae(buf["rew"], values, buf["done"].astype(np.float64), gamma, lam)     # Ψ_t = Â_t
        batch = {"obs": buf["obs"].flatten(0, 1), "act": buf["act"].flatten(0, 1), "logp": buf["logp"].flatten(),
                 "adv": torch.as_tensor(adv, dtype=torch.float32).flatten(),
                 "ret": torch.as_tensor(adv + values[:-1], dtype=torch.float32).flatten()}
        kl, clipfrac = ppo_update(agent, opt, batch, epochs, minibatch_size, clip, vf_coef, ent_coef, max_grad_norm)
        kls.append(kl)
        clipfracs.append(clipfrac)
        while ep["global_step"] >= next_eval:                  # evaluate on the same step grid as the other methods
            evals.append((next_eval, evaluate(lambda o: agent.act(o), seed=10**6 + next_eval)))
            next_eval += eval_every
    envs.close()
    return {"seed": seed, "returns": ep["returns"], "steps_at": ep["steps_at"], "evals": evals,
            "approx_kl": kls, "clipfrac": clipfracs, "agent": agent.snapshot(),
            "reward_scale": float(np.sqrt(ret_rms.var + 1e-8))}

## 6.7 Training and all four methods compared

One run per seed, in parallel as before, with the same budget of environment steps as Parts 2 to 4. Then the
whole story on one plot, and the per-seed table with the count of seeds on which every method beat the one before.

In [ ]:
t0 = time.perf_counter()
ppo_runs = util.parallel_map(lambda seed: ppo(seed, PPO_TOTAL_STEPS, eval_every=EVAL_EVERY), SEEDS)
print(f"{len(SEEDS)} seeds × {PPO_TOTAL_STEPS:,} steps in {time.perf_counter() - t0:.0f} s")

plot_methods({"REINFORCE": (reinforce_runs, AMBER), "with a baseline": (baseline_runs, GREEN),
              "actor-critic": (ac_runs, PURPLE2), "PPO": (ppo_runs, PURPLE)},
             "All four methods on LunarLander, same seeds, same steps")

env, clips, labels = make_env(render_mode="rgb_array"), [], []
for run in ppo_runs:
    agent = from_snapshot(run["agent"])
    frames, ret = util.record_episode(env, lambda o: agent.act(o), seed=123, frame_every=3)
    clips.append(frames)
    labels.append(f"seed {run['seed']}: return {ret:.0f}")
display(util.show_video(util.tile(clips, labels, cols=min(3, len(clips)), gap=6), "ppo.mp4", fps=17, width=960))
RESULTS["PPO"] = float(np.mean([final_score(r) for r in ppo_runs]))

names = ["REINFORCE", "with baseline", "actor-critic", "PPO"]
all_runs = (reinforce_runs, baseline_runs, ac_runs, ppo_runs)
print(f"{'':>8}" + "".join(f"{n:>15}" for n in names) + "   (final score: mean of the last three evaluations)")
for i, seed in enumerate(SEEDS):
    print(f"{seed:>8}" + "".join(f"{final_score(runs[i]):>15.0f}" for runs in all_runs))
better = [all(final_score(all_runs[k + 1][i]) > final_score(all_runs[k][i]) for k in range(3)) for i in range(len(SEEDS))]
print(f"\neach method beats the one before it on {sum(better)} of {len(SEEDS)} seeds")

## 6.8 🔬 Experiment: how many steps on one batch

PPO's main new knob. The number of epochs sets $K$, how many gradient steps each batch serves; the clip $\epsilon$,
kept at 0.2, limits how far the policy may move on it.

**Hypotheses to check.** (1) One epoch ($K = 32$) wastes the batch and learns slowly. (2) Thirty epochs ($K = 960$)
might overfit each batch; or does the clip keep them safe? (3) Somewhere between, the knob stops mattering much —
PPO is known for being forgiving.

In [ ]:
epochs_runs = experiment(ppo, {"1 epoch (K = 32)": dict(epochs=1), "3 epochs (K = 96)": dict(epochs=3),
                               "10 epochs (K = 320, ours)": dict(epochs=10), "30 epochs (K = 960)": dict(epochs=30)},
                         "PPO: gradient steps per batch, same budget of environment steps")

## 6.9 🔬 Experiment: the dials of GAE, $\gamma$ and $\lambda$

Part 5 explained $\lambda$ as a dial between actor-critic's $\delta_t$ and the baseline's $G_t - V_\phi$, and the
discount $\gamma$ as how far ahead the agent looks. Here both are turned inside PPO.

**Questions to answer from the plot.** (1) Does $\lambda = 0$ (one-step errors) learn worse than $\lambda = 0.95$,
as the critic's errors enter every multiplier? (2) Is $\lambda = 1$ (no bootstrapping) noisier, and by how much?
(3) $\gamma = 0.9$ makes the agent short-sighted: a landing takes a few hundred steps, and $0.9^{100} \approx 0$, so
the +100 for coming to rest barely reaches the approach. Does it still land? (4) $\gamma = 0.999$ makes the returns
larger and longer; does the critic cope, with the reward scaling of 6.3 to help it?

In [ ]:
lam_runs = experiment(ppo, {"λ = 0 (one-step errors)": dict(lam=0.0), "λ = 0.8": dict(lam=0.8),
                            "λ = 0.95 (ours)": dict(lam=0.95), "λ = 1 (no bootstrapping)": dict(lam=1.0)},
                      "PPO: GAE's λ")

In [ ]:
gamma_runs = experiment(ppo, {"γ = 0.9": dict(gamma=0.9), "γ = 0.99 (ours)": dict(gamma=0.99), "γ = 0.999": dict(gamma=0.999)},
                        "PPO: the discount γ")

**Reading the experiments.** Two of these dials, $\gamma$ and $\lambda$, belong to the *estimate of the gradient*;
the third, $K$, to *how the estimate is used*. The lecture's order of ideas is the order of the
dials: first a better multiplier, then a safer way to take steps on it.

## ⭐ Bonus 5 — The gradient of the clipped objective, and the spread of $\hat A_t$

*Nothing later depends on this cell.*

**(a)** Derive $\nabla_w L$ on paper, then write your result into `my_gradient` in the first cell below: it
compares your derivative with autograd's at four ratios and both signs of $\Psi_t$. At which ratios does it vanish, and
why does the sign of $\Psi_t$ decide on which side it vanishes?

**(b)** Part 5's table says the noise of $\hat A_t$ grows with $\lambda$. Measure it on a trained agent: the second cell
below plays 10 episodes with the stochastic policy of `ppo_runs[0]`, computes $\hat A_t$ for several $\lambda$ with
its critic, and plots the spread. Two effects pull against each other — the luck of later steps grows with
$\lambda$, while the critic's own error at each step is averaged out by longer sums — so which $\lambda$ has the
smallest spread depends on how good the critic is.

In [ ]:
# ⭐ Bonus 5 (a) — a starting point: your derivative of the clipped objective against autograd. Fill `my_gradient`.
RUN_BONUS = False
if RUN_BONUS:
    def my_gradient(ratio: float, psi: float, clip: float = 0.2) -> float:
        "Your derivative of one term of minus L, -min(ρΨ, clip(ρ)Ψ), with respect to the new log-probability log π_w."
        # 🎯 from your derivation on paper: which branch of the min is active, and what is d ρ / d log π_w?
        return ...

    must_fill(_where="this cell", my_gradient=my_gradient(1.0, 1.0))
    print("Ψ_t     ρ_t    autograd   yours")
    for psi in (+1.0, -1.0):
        for ratio in (0.7, 0.9, 1.1, 1.3):
            new_logp = torch.tensor([math.log(ratio)], requires_grad=True)
            ppo_policy_loss(new_logp, torch.zeros(1), torch.tensor([psi]), 0.2).backward()
            auto, yours = new_logp.grad.item(), float(my_gradient(ratio, psi))
            print(f"{psi:+.0f}     {ratio:.1f}   {auto:+7.3f}   {yours:+7.3f}   {'✓' if abs(auto - yours) < 1e-6 else '✗'}")

In [ ]:
# ⭐ Bonus 5 (b) — the spread of Â_t for each λ, on the trained agent of the first seed.
RUN_BONUS = False
if RUN_BONUS:
    def advantages_by_lambda(agent, lams, reward_scale, n_episodes=10, gamma=0.99, seed=2000):
        out = {lam: [] for lam in lams}
        env = make_env()
        for i in range(n_episodes):
            obs, _ = env.reset(seed=seed + i)
            states, rewards = [obs], []
            while True:
                obs, r, term, trunc, _ = env.step(agent.act(obs))
                states.append(obs)
                rewards.append(r / reward_scale)                # the critic learned scaled rewards
                if term or trunc:
                    break
            with torch.no_grad():
                values = agent.value(agent.normalise(np.stack(states))).numpy()
            dones = np.zeros(len(rewards))
            dones[-1] = float(term)
            for lam in lams:
                out[lam].append(compute_gae(np.array(rewards), values, dones, gamma, lam))
        return out

    LAMS = [0.0, 0.5, 0.8, 0.9, 0.95, 0.99, 1.0]
    adv_by_lam = advantages_by_lambda(from_snapshot(ppo_runs[0]["agent"]), LAMS, ppo_runs[0]["reward_scale"])
    spread = {lam: float(np.std(np.concatenate(adv_by_lam[lam]))) for lam in LAMS}
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(LAMS, [spread[lam] for lam in LAMS], "o-", color=PURPLE2)
    ax.set_xlabel("λ"); ax.set_ylabel("standard deviation of Â_t"); ax.set_title(f"seed {ppo_runs[0]['seed']}")
    plt.show()
TIMINGS["6 PPO"] = time.perf_counter() - PART_T0

# Summary

The wall-clock time of every part on this machine, and the final score of each method (mean over the seeds).

In [ ]:
print("seeds:", SEEDS)
print("\nWall-clock time per part")
for k, v in TIMINGS.items():
    print(f"  {k:28s} {v:8.1f} s")
print(f"  {'total':28s} {sum(TIMINGS.values()):8.1f} s\n")
print(f"Final score, mean over the seeds ({SOLVED} or more is solved)")
for k, v in RESULTS.items():
    print(f"  {k:45s} {v:6.0f}")